## Custom Resource Management: Writing Context Managers
- Whenever you need custom setup/teardown logic, you can write your own Context Manager.
- A context manager ensures that teardown always runs, even if errors occur in the block.  
- Two approaches: implement `__enter__`/`__exit__` in a class or use the simpler generator-based decorator.  

In [8]:
class MyContextManager:
    def __init__(self, timeout):
        self.timeout = timeout

    def __enter__(self):
        print("Setup complete")
        return "a simple statement"
    
    def __exit__(self, *args): #exc_type, exc, tb
        print(f"Teardown")

        # print(exc_type, exc, tb)
        for ar in args:
            print(ar)

        return False
        
with MyContextManager(timeout=30) as cm:
    print(cm)
    print("Inside the block")
    raise ValueError("Simulated Problem")

Setup complete
a simple statement
Inside the block
Teardown
<class 'ValueError'>
Simulated Problem


ValueError: Simulated Problem

## The `@contextlib.contextmanager` Decorator
- Provided by the `contextlib` module to turn a generator into a context manager.  
- Decorated function needs exactly one `yield`.  
- Code before `yield` runs as `__enter__`; code after (or in `finally`) runs as `__exit__`.  
- Simplifies many common patterns without writing a full class.

###  Generator Structure for `@contextmanager`
- Wrap the `yield` in `try...finally` to ensure teardown even on errors.  
- The value yielded is bound to `as var` in the `with` statement (if used).  
- You can catch exceptions inside the generator if you want to suppress them.

In [8]:
import os
from contextlib import contextmanager

@contextmanager
def change_directory(destination):
    """
    Temporarily switch into destination. If the directory does not exist, 
    it is creted before just the switch

    Args:
        destination (str): Path to the destination that should be become teh working directory.
    """

    original_dir = os.getcwd()

    try:
        print(f"Changing in {destination}")
        os.makedirs(destination, exist_ok=True)
        os.chdir(destination)
        yield os.getcwd()
    finally:
        print(f"Reverting into the original dir: {original_dir}")
        os.chdir(original_dir)

print(f"Start: {os.getcwd()}")

with change_directory("temp_dir") as new_dir:
    print(f"Inside: {new_dir}")

print(f"End: {os.getcwd()}")



Start: c:\Users\islam\Documents\Programming\PythonProgramming\python-devops-start\error-handling
Changing in temp_dir
Inside: c:\Users\islam\Documents\Programming\PythonProgramming\python-devops-start\error-handling\temp_dir
Reverting into the original dir: c:\Users\islam\Documents\Programming\PythonProgramming\python-devops-start\error-handling
End: c:\Users\islam\Documents\Programming\PythonProgramming\python-devops-start\error-handling
